# Chirp v3 Batch Transcription Pipeline

This Colab orchestrates the large-scale transcription of audio segments using Google Cloud's **Chirp v3** (Speech-to-Text v2) model.

### Workflow Overview:
1.  **Job Submission**: Reads a JSONL manifest from GCS and submits asynchronous batch jobs in chunks of 15 segments. It automatically filters out files without speech to optimize API costs.
2.  **Organization**: Results are written to a timestamped directory structure in GCS (`transcripts/{timestamp}/batch_n/`) for easy tracking and prevention of file collisions.
3.  **Real-time Monitoring**: Provides a lightweight polling dashboard that shows batch progress, server update times, and a bulleted list of truly outstanding files by cross-referencing existing result blobs in GCS.
4.  **Error Diagnostics**: Includes a diagnostic tool to scan completed operations for file-level rejections or API errors.

In [18]:
# @title Install dependencies
!pip install loguru

Note: This Colab reads/writes content in the `wd-asr-chirp-evaluation` bucket in GCP.

In [19]:
# @title Import dependencies
from google.colab import auth

from google.api_core import client_options
from google.api_core.client_options import ClientOptions
from google.cloud import storage
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech

from loguru import logger

import json
import os
import time
import sys

from tqdm.auto import tqdm

from IPython.display import clear_output

In [20]:
# @title Define constants and initial logging
GCP_PROJECT_ID = "automatic-hawk-481415-m9"
GCS_BUCKET="wd-asr-chirp-evaluation"
GCS_INPUT_DIR="segmented_audio"
GCS_OUTPUT_DIR="transcripts"

# Corrected GCS manifest path
MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"

# Initialize loguru
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}");

In [21]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

Updated property [core/project].


**NOTE:** I needed to add needed access to the bucket for Chirp to access the files for batch processing.
```
!gsutil iam ch serviceAccount:service-781667204380@gcp-sa-speech.iam.gserviceaccount.com:objectAdmin gs://$GCS_BUCKET
```

In [22]:
def start_chirp_jobs_from_manifest(
    manifest_gcs_uri: str, project_id: str = GCP_PROJECT_ID, overwrite: bool = False
) -> tuple[list[str], int, dict[str, list[str]]]:
    """
    Reads the batch JSONL manifest from GCS and submits transcription jobs.
    """
    opts = client_options.ClientOptions(
        api_endpoint="us-speech.googleapis.com", quota_project_id=project_id
    )
    client = SpeechClient(client_options=opts)
    storage_client = storage.Client(project=project_id)

    batch_timestamp = int(time.time())

    # 1. Download and Parse the manifest from GCS
    logger.info(f"Reading manifest from GCS: {manifest_gcs_uri}")
    bucket_name = manifest_gcs_uri.replace("gs://", "").split("/")[0]
    blob_path = "/".join(manifest_gcs_uri.replace("gs://", "").split("/")[1:])

    try:
        content = storage_client.bucket(bucket_name).blob(blob_path).download_as_text()
    except Exception as e:
        logger.error(f"Failed to download manifest: {e}")
        return [], batch_timestamp, {}

    manifest_entries = []
    for line in content.strip().split("\n"):
        if line.strip():
            manifest_entries.append(json.loads(line))

    logger.info(f"Manifest contains {len(manifest_entries)} total entries.")

    if not manifest_entries:
        return [], batch_timestamp, {}

    # 2. Handle Overwrite
    if overwrite:
        logger.info("Overwrite requested. Clearing existing transcripts in GCS...")
        bucket = storage_client.bucket(GCS_BUCKET)
        blobs_to_delete = list(storage_client.list_blobs(GCS_BUCKET, prefix=f"{GCS_OUTPUT_DIR}/"))
        if blobs_to_delete:
            bucket.delete_blobs(blobs_to_delete)
        existing_basenames = set()
    else:
        existing_blobs = list(storage_client.list_blobs(GCS_BUCKET, prefix=f"{GCS_OUTPUT_DIR}/"))
        existing_basenames = {os.path.basename(b.name) for b in existing_blobs if b.name.endswith(".json")}

    # 3. Identify pending files
    pending_audio_uris = []
    for entry in manifest_entries:
        uri = entry["audio_filepath"]
        fname_base = os.path.basename(uri).replace(".flac", "")

        if not overwrite and any(fname_base in res_name for res_name in existing_basenames):
            continue

        pending_audio_uris.append(uri)

    logger.info(f"Identified {len(pending_audio_uris)} pending audio URIs for submission.")

    if not pending_audio_uris:
        return [], batch_timestamp, {}

    # 4. Chunk and Submit
    batch_size = 15
    chunks = [pending_audio_uris[i : i + batch_size] for i in range(0, len(pending_audio_uris), batch_size)]

    operation_names = []
    op_to_filenames = {}

    for i, segment_batch in enumerate(chunks):
        target_output = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_DIR}/"
        request = cloud_speech.BatchRecognizeRequest(
            recognizer=f"projects/{project_id}/locations/us/recognizers/_",
            config=cloud_speech.RecognitionConfig(
                auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
                model="chirp_3",
                language_codes=["en-US"],
                features=cloud_speech.RecognitionFeatures(enable_word_time_offsets=True, enable_automatic_punctuation=True),
            ),
            files=[cloud_speech.BatchRecognizeFileMetadata(uri=u) for u in segment_batch],
            recognition_output_config=cloud_speech.RecognitionOutputConfig(
                gcs_output_config=cloud_speech.GcsOutputConfig(uri=target_output)
            ),
        )

        try:
            operation = client.batch_recognize(request=request)
            op_name = operation._operation.name
            operation_names.append(op_name)
            op_to_filenames[op_name] = segment_batch
            logger.info(f"Started Batch {i}: {op_name.split('/')[-1]}")
        except Exception as e:
            logger.warning(f"Failed to submit Batch {i}: {e}")

    return operation_names, batch_timestamp, op_to_filenames


def poll_all_jobs(
    operation_names: list[str],
    batch_timestamp: int,
    op_to_filenames: dict[str, list[str]] | None = None,
    project_id: str = GCP_PROJECT_ID,
) -> None:
    opts = client_options.ClientOptions(
        api_endpoint="us-speech.googleapis.com", quota_project_id=project_id
    )
    client = SpeechClient(client_options=opts)
    storage_client = storage.Client(project=project_id)
    op_client = client.transport.operations_client

    reference_start_time = None

    while True:
        clear_output(wait=True)
        all_done = True

        if reference_start_time is None and operation_names:
            try:
                first_op = op_client.get_operation(name=operation_names[0])
                meta = cloud_speech.OperationMetadata.deserialize(
                    first_op.metadata.value
                )
                reference_start_time = meta.create_time.timestamp()
            except:
                reference_start_time = time.time()

        elapsed_hms = time.strftime(
            "%H:%M:%S", time.gmtime(int(time.time() - reference_start_time))
        )
        current_time = time.strftime("%H:%M:%S")

        logger.info(f"--- Chirp Status | Batch ID: {batch_timestamp} ---")
        logger.info(f"(Elapsed: {elapsed_hms} | Now: {current_time} UTC)\n")

        # Check root transcripts folder
        blobs = list(storage_client.list_blobs(GCS_BUCKET, prefix=f"{GCS_OUTPUT_DIR}/"))
        existing_json_basenames = {
            os.path.basename(b.name) for b in blobs if b.name.endswith(".json")
        }

        for i, name in enumerate(operation_names):
            op = op_client.get_operation(name=name)
            batch_uris = op_to_filenames.get(name, []) if op_to_filenames else []

            if op.done:
                if op.error.message:
                    logger.info(f"Batch {i}: Failed - {op.error.message}")
                else:
                    logger.info(f"Batch {i}: Complete ({len(batch_uris)} Segments)")
            else:
                all_done = False
                metadata = cloud_speech.OperationMetadata.deserialize(op.metadata.value)
                progress = metadata.progress_percent
                last_upd = metadata.update_time.strftime("%H:%M:%S")
                logger.info(f"Batch {i}: {progress}% (Server Update: {last_upd})")

                pending_files = []
                for uri in batch_uris:
                    fname = os.path.basename(uri)
                    fname_base = fname.replace(".flac", "")
                    if not any(
                        fname_base in res_name for res_name in existing_json_basenames
                    ):
                        pending_files.append(fname)

                if pending_files:
                    for f in pending_files[:5]:
                        logger.info(f"  • {f}")
                    if len(pending_files) > 5:
                        logger.info(f"  • ... and {len(pending_files) - 5} more")

        if all_done:
            logger.info("\nAll batches finished!")
            break

        time.sleep(15)

def diagnose_failed_files(operation_names, project_id=GCP_PROJECT_ID):
    opts = ClientOptions(
        api_endpoint="us-speech.googleapis.com", quota_project_id=project_id
    )
    client = SpeechClient(client_options=opts)
    op_client = client.transport.operations_client

    logger.info("Scanning batch operations for file-level errors...\n")

    total_failed = 0

    for name in operation_names:
        op = op_client.get_operation(name=name)

        if not op.done:
            logger.info(f"Batch {name.split('/')[-1]} is still running, skipping...")
            continue

        if op.error.message:
            logger.info(f"Entire Batch Failed: {op.error.message}")
            continue

        if op.response:
            response = cloud_speech.BatchRecognizeResponse.deserialize(
                op.response.value
            )

            for uri, file_result in response.results.items():
                if file_result.error.code != 0:
                    total_failed += 1
                    logger.info(f"   Failed File: {uri}")
                    logger.info(f"   Error Code: {file_result.error.code}")
                    logger.info(f"   Message: {file_result.error.message}\n")

    if total_failed == 0:
        logger.info(
            "No file-level errors found in the completed batches."
        )
    else:
        logger.info(f"Found {total_failed} total files that Chirp rejected.")

In [23]:
# @title Execute batch jobs
# Re-running with the corrected filenames in GCS
operations, timestamp, op_to_filenames = start_chirp_jobs_from_manifest(MANIFEST_URI, overwrite=True)

logger.info(f"\n--- Batch Submission Summary ---")
logger.info(f"Batch ID: {timestamp}")
logger.info(f"Total Operations started: {len(operations)}")
logger.info(f"Results are being written to: gs://{GCS_BUCKET}/{GCS_OUTPUT_DIR}/")

INFO: Reading manifest from GCS: gs://wd-asr-chirp-evaluation/segmented_audio/batch_manifest.jsonl
INFO: Manifest contains 40 total entries.
INFO: Overwrite requested. Clearing existing transcripts in GCS...
INFO: Identified 40 pending audio URIs for submission.
INFO: Started Batch 0: v2-69e87971-0000-2422-b5cf-d4f547f91cc8
INFO: Started Batch 1: v2-6a207699-0000-2ef0-9cb7-d4f547ed6e7c
INFO: Started Batch 2: v2-6a1d568c-0000-25df-ae5f-d4f547f49020
INFO: 
--- Batch Submission Summary ---
INFO: Batch ID: 1772733014
INFO: Total Operations started: 3
INFO: Results are being written to: gs://wd-asr-chirp-evaluation/transcripts/


In [24]:
# @title Poll progress on batch jobs
if operations:
    poll_all_jobs(operations, timestamp, op_to_filenames)
else:
    logger.info("No new operations were started.")

INFO: --- Chirp Status | Batch ID: 1772733014 ---
INFO: (Elapsed: 00:01:03 | Now: 17:51:18 UTC)

INFO: Batch 0: Complete (15 Segments)
INFO: Batch 1: Complete (15 Segments)
INFO: Batch 2: Complete (10 Segments)
INFO: 
All batches finished!


In [25]:
# @title Check for any failures
if operations:
    diagnose_failed_files(operations)
else:
    logger.info("No 'operations' list found. You may need to re-run your batch cell or paste the operation strings here.")

INFO: Scanning batch operations for file-level errors...

INFO: No file-level errors found in the completed batches.
